### Dynamisches Programmieren

[Video Einführung](https://youtu.be/WjOtEQ2_aQI) - [Video Frog 1](https://youtu.be/Wv5LPaB1iok) - [Video Cooldown](https://youtu.be/4NT1VjjeeO8) -
[Video Tabulation](https://youtu.be/ZC_JCyyb1OE) - [Video Parent Pointer](https://youtu.be/L_kNyw99YKc) -
[alle Videos](https://www.youtube.com/playlist?list=PLWeMgMhRDsIFAat7gmQlv95f2dl0sUqPq)

Dynamisches Programmieren können wir häufig beschreiben als eine Verbindung von **Rekursion**, **Memoization** und ein
bisschen **brute force**:

- Rekursion: Wir lösen ein Problem mit Hilfe der Lösungen für kleinere Problemgrößen.
- brute force: Was wir für die Rekursion nicht wissen, probieren wir einfach in allen Möglichkeiten aus.
- Memoization: Jedes Teilproblem wird nur einmal berechnet, das Ergebnis merken wir uns.

Die Beispiele in diesem Notebook stammen meist aus Programmierwettbewerben (atcoder, leetcode).

#### Frog 1

https://atcoder.jp/contests/dp/tasks/dp_a

Gegeben eine Liste a:

    a = [30, 10, 60, 10, 60, 50]

Der Frosch sitzt auf der ersten Zahl und will zur letzten hüpfen. Er kann immer nur 1 oder 2 Zahlen weiter hüpfen. Die
Kosten für einen Sprung sind der Unterschied der beiden Zahlen, zwischen denen er springt. Ziel ist es, die Kosten zu
minimieren.

<img src='frog1.png' width='500'>

Der blaue Weg kostet $20 + 0 + 40 = 60$, der grüne nur $30 + 0 + 10 = 40$.

#### Teilprobleme

Bei der Rekursion dürfen wir für die Lösung des Problems die Antworten für kleinere Problemgrößen benutzen. Bezieht
sich das Problem auf eine Liste `a`, dann versuchen wir es zunächst mit **Suffixen**, also Teillisten `a[i:]`.

<img src='dp2.png' width='600'>

    n = len(a)
    dp(i) = Lösung des Problems für a[i:],  i = 0..n-1 oder i = 0..n
    dp(0) = Lösung des Originalproblems

Für die Rekursionsgleichung müssen wir `dp(i)` durch `dp`-Werte mit größerem Index ausdrücken, denn die gehören zu
kleineren Problemgrößen:

    dp(i) = .... dp(i+1) ... dp(i+2)

Oft können wir die Rekursionsgleichung nicht ohne Weiteres aufstellen. Dann fragen wir uns: Was würden wir von der
Lösung `dp(i)` gerne wissen? Diese Frage nennen wir **dp-guess**. Bei Suffixen bezieht sie sich meist auf das, was am
Anfang gemacht wird. Mit einer Antwort auf die Frage muss es rekursiv weitergehen. Dann probieren wir alle möglichen
Antworten aus (brute force) und nehmen die beste.

Beim Frosch fragen wir: Wie groß ist der erste Sprung? Wir wissen es nicht, also probieren wir beide Sprünge aus:

<img src='dp3.png' width='700'>

Die **Rekursionsbremse** ist das kleinste Teilproblem, das wir direkt lösen: Steht der Frosch schon auf der letzten Zahl,
kostet es nichts mehr. Steht er auf der vorletzten, kann er nur noch einen Sprung machen.

Zuerst die Rekursion ohne Memoization:

In [1]:
def dp(i):
    '''
    returns: die minimalen Kosten für a[i:], i = 0..n-1
    dp-guess: wie groß ist der erste Sprung?
    '''
    if i == n - 1:
        return 0
    if i == n - 2:
        return abs(a[n - 1] - a[n - 2])
    return min(abs(a[i] - a[i + 1]) + dp(i + 1), abs(a[i] - a[i + 2]) + dp(i + 2))

a = [30, 10, 60, 10, 60, 50]
n = len(a)
print(dp(0))

40


Das ist richtig, aber langsam: `dp(i + 1)` ruft `dp(i + 3)` auf, `dp(i + 2)` auch, und so weiter. Dieselben Teilprobleme
werden immer wieder berechnet. Deshalb merken wir uns jedes Ergebnis im Dictionary `memo` und schauen zuerst dort nach.
Wir zählen die Aufrufe für eine Liste mit 25 Zahlen:

In [2]:
def dp_ohne_memo(i):
    global aufrufe
    aufrufe += 1
    if i == n - 1:
        return 0
    if i == n - 2:
        return abs(a[n - 1] - a[n - 2])
    return min(abs(a[i] - a[i + 1]) + dp_ohne_memo(i + 1), abs(a[i] - a[i + 2]) + dp_ohne_memo(i + 2))

def dp(i):
    '''
    returns: die minimalen Kosten für a[i:], i = 0..n-1
    dp-guess: wie groß ist der erste Sprung?
    '''
    global aufrufe
    aufrufe += 1
    if i in memo:
        return memo[i]
    if i == n - 1:
        return 0
    if i == n - 2:
        return abs(a[n - 1] - a[n - 2])
    res = min(abs(a[i] - a[i + 1]) + dp(i + 1), abs(a[i] - a[i + 2]) + dp(i + 2))
    memo[i] = res
    return res

import random
random.seed(1)
a = [random.randint(1, 100) for i in range(25)]
n = len(a)

aufrufe = 0
print(dp_ohne_memo(0), aufrufe, 'Aufrufe ohne Memoization')
aufrufe = 0
memo = {}
print(dp(0), aufrufe, 'Aufrufe mit Memoization')

379 150049 Aufrufe ohne Memoization
379 47 Aufrufe mit Memoization


Ohne Memoization wächst die Zahl der Aufrufe wie die Fibonacci-Zahlen, also exponentiell. Mit Memoization wird jedes
Teilproblem nur einmal berechnet.

**Laufzeit** = Anzahl der Teilprobleme · nicht-rekursive Arbeit pro Teilproblem. Die rekursiven Aufrufe dürfen wir
wegen der Memoization mit $O(1)$ ansetzen. Beim Frosch sind es $n$ Teilprobleme mit konstanter Arbeit, die Laufzeit
ist $O(n)$.

##### Welche Indizes darf man benutzen?

Je nachdem, welche Fälle die Rekursionsbremse abfängt, dürfen wir in der Rekursion bestimmte Indizes ohne weitere
Prüfung verwenden:

- Bremse bei `i == n`: In der Rekursion sind `a[i]` und `dp(i + 1)` sicher.
- Bremse bei `i == n - 1`: Dann sind auch `a[i + 1]` und `dp(i + 2)` sicher.
- Brauchen wir höhere Indizes, müssen wir sie in der Rekursion prüfen (z.B. `i + 2 < n`) oder die Bremse um weitere
  Fälle erweitern, wie oben bei `i == n - 2`.

Hat das Problem für die leere Liste eine sinnvolle Lösung, nimmt man meist `i == n` als Bremse.

#### Frog 2

https://atcoder.jp/contests/dp/tasks/dp_b

Der Frosch sitzt auf der ersten Zahl und will zur letzten hüpfen. Er kann jetzt 1 bis k Zahlen weit hüpfen. Die Kosten
für einen Sprung sind wieder der Unterschied der Zahlen. Ziel ist es, die Kosten zu minimieren.

<img src='frog2.png' width='600'>

Der dp-guess ist derselbe, aber jetzt gibt es k Möglichkeiten für den ersten Sprung. Sprünge über das Ende hinaus
schließen wir mit `i + j < n` aus.

In [3]:
def dp(i):
    '''
    returns: die minimalen Kosten für a[i:]
    dp-guess: wie groß ist der erste Sprung?
    '''
    if i in memo:
        return memo[i]
    if i == n - 1:
        return 0
    res = abs(a[i] - a[i + 1]) + dp(i + 1)          # erster Sprung 1
    for j in range(2, k + 1):                       # geht es mit erstem Sprung 2..k besser?
        if i + j < n and abs(a[i] - a[i + j]) + dp(i + j) < res:
            res = abs(a[i] - a[i + j]) + dp(i + j)
    memo[i] = res
    return res

k = 4
a = [40, 10, 20, 70, 80, 10, 20, 70, 80, 60]
n = len(a)
memo = {}
print(dp(0))

40


Laufzeit: $n$ Teilprobleme mit je $O(k)$ Arbeit, also $O(k \cdot n)$.

#### Bowling

Gegeben sind n Bowling-Kegel in einer Reihe. Mit einem Wurf können wir 0, 1 oder 2 nebeneinanderstehende Kegel treffen.
Kegel i hat den Wert `a[i]`. Trifft man einen Kegel, bekommt man `a[i]` Punkte. Trifft man zwei, bekommt man das
Produkt ihrer Werte. Man muss nicht alle treffen. Ziel ist es, die Punkte zu maximieren.

<img src='bowling.png' width='601'>

dp-guess: Was wird mit dem ersten Kegel gemacht? Stehen gelassen, allein getroffen oder zusammen mit dem nächsten?

In [4]:
def dp(i):
    '''
    returns: maximale Punktzahl für die Liste a[i:], i = 0..n
    dp-guess: was wird mit dem ersten Kegel gemacht: stehen gelassen, einzeln oder im Paar getroffen?
    '''
    if i in memo:
        return memo[i]
    if i == n:
        return 0
    if i == n - 1:
        return max(a[i], 0)
    res = max(dp(i + 1), a[i] + dp(i + 1), a[i] * a[i + 1] + dp(i + 2))
    memo[i] = res
    return res

a = [1, 1, 9, 9, 2, -5, -5]
n = len(a)
memo = {}
print(dp(0))

110


Hier ist `i == n` die Bremse, denn für keine Kegel gibt es eine sinnvolle Lösung: 0 Punkte. Weil wir `a[i + 1]`
benutzen, fangen wir zusätzlich `i == n - 1` ab. Laufzeit $O(n)$.

Die beste Lösung: 1 und 1 einzeln (2 Punkte), 9 und 9 als Paar (81), 2 einzeln (2), -5 und -5 als Paar (25), zusammen
110.

#### Cooldown

Die Liste `a` zeigt die Wertentwicklung einer Aktie an aufeinanderfolgenden Tagen. Wir können an einem Tag entweder
kaufen, verkaufen oder nichts machen. Wir können höchstens eine Aktie besitzen. Wenn wir verkauft haben, können wir am
nächsten Tag nicht sofort wieder kaufen (cooldown). Welcher maximale Gewinn ist möglich?

<img src='cooldown.png' width='500'>

<img src='cooldown1.png' width='600'>

Was wir am Tag i machen dürfen, hängt davon ab, ob wir eine Aktie haben und ob wir im Cooldown sind. Deshalb führen wir
einen zusätzlichen Parameter `k` für den **Zustand** ein:

<img src='cooldown2.png' width='600'>

In [5]:
def dp(i, k):
    '''
    returns: maximaler Gewinn für a[i:], wenn wir am Tag i im Zustand k sind, i = 0..n
        k = 0: keine Aktie vorhanden, nicht im Cooldown
        k = 1: keine Aktie vorhanden, im Cooldown
        k = 2: Aktie vorhanden
    dp-guess: was machen wir am Tag i im Zustand k?
    '''
    if (i, k) in memo:
        return memo[(i, k)]
    if i == n:
        return 0
    if k == 0:
        res = max(dp(i + 1, 0), dp(i + 1, 2) - a[i])        # nicht kaufen oder kaufen
    elif k == 1:
        res = dp(i + 1, 0)                                  # Cooldown
    else:
        res = max(dp(i + 1, 2), dp(i + 1, 1) + a[i])        # nicht verkaufen oder verkaufen
    memo[(i, k)] = res
    return res

a = [1, 2, 3, 0, 2]
n = len(a)
memo = {}
dp(0, 0)

3

Am Anfang haben wir keine Aktie und sind nicht im Cooldown, deshalb ist `dp(0, 0)` die Lösung: am Tag 0 für 1 kaufen,
am Tag 1 für 2 verkaufen, Cooldown, am Tag 3 für 0 kaufen und am Tag 4 für 2 verkaufen, Gewinn $1 + 2 = 3$.

Es gibt $3n$ Teilprobleme mit konstanter Arbeit, die Laufzeit ist $O(n)$.

Häufig ist ein Teilproblem dasselbe Problem wie das Original, nur kleiner. Manchmal brauchen wir, wie hier, eine
zusätzliche Information, die die Möglichkeiten einschränkt. Dann bekommt `dp` einen weiteren Parameter.

#### Vacation

https://atcoder.jp/contests/dp/tasks/dp_c

Gegeben sind drei Listen a, b und c der Länge n. Für jeden Index wählen wir eine Zahl aus einer der drei Listen. Es
dürfen nicht zweimal nacheinander Zahlen aus derselben Liste gewählt werden. Was ist die maximale Summe der gewählten
Zahlen?

<img src='vacation.png' width='400'>

Wenn wir an Stelle i beginnen, können wir jede der drei Listen wählen, aber die rekursiven Aufrufe dürfen nicht
dieselbe Liste wieder wählen. Deshalb geben wir als Parameter mit, welche Liste an der Stelle i gewählt wird.

    dp(i, k) = die maximale Summe, wenn wir am Index i beginnen und dort aus Liste k wählen
               (k = 0: Liste a, k = 1: Liste b, k = 2: Liste c)

Das Ausgangsproblem ist dann das Maximum von `dp(0, 0)`, `dp(0, 1)` und `dp(0, 2)`.

In [6]:
def dp(i, k):
    '''
    returns: maximale Summe, wenn wir am Index i beginnen und dort aus Liste k wählen, i = 0..n
    dp-guess: welche Liste wird als Nächstes gewählt?
    '''
    if (i, k) in memo:
        return memo[(i, k)]
    if i == n:
        return 0
    if k == 0:
        res = a[i] + max(dp(i + 1, 1), dp(i + 1, 2))
    if k == 1:
        res = b[i] + max(dp(i + 1, 0), dp(i + 1, 2))
    if k == 2:
        res = c[i] + max(dp(i + 1, 0), dp(i + 1, 1))
    memo[(i, k)] = res
    return res

a = [10, 20, 30]
b = [40, 50, 60]
c = [70, 80, 90]
n = len(a)
memo = {}
print(max(dp(0, 0), dp(0, 1), dp(0, 2)))

210


Laufzeit $O(n)$. Man kann den Parameter auch anders deuten: `k` ist die Liste, die an der Stelle i **nicht** gewählt
werden darf, weil sie davor gewählt wurde.

In [7]:
def dp(i, k):
    '''
    returns: maximale Summe ab Index i, wenn dort aus Liste k nicht gewählt werden darf, i = 0..n
    '''
    if (i, k) in memo:
        return memo[(i, k)]
    if i == n:
        return 0
    if k == 0:
        res = max(dp(i + 1, 1) + b[i], dp(i + 1, 2) + c[i])
    if k == 1:
        res = max(dp(i + 1, 0) + a[i], dp(i + 1, 2) + c[i])
    if k == 2:
        res = max(dp(i + 1, 0) + a[i], dp(i + 1, 1) + b[i])
    memo[(i, k)] = res
    return res

memo = {}
print(max(dp(0, 0), dp(0, 1)))

210


Am Anfang ist keine Liste verboten. `dp(0, 0)` probiert b und c, `dp(0, 1)` probiert a und c. Zusammen sind damit alle
drei Listen für den Index 0 abgedeckt.

### Tabulation

Python begrenzt die Rekursionstiefe. Die Grenze kann man erhöhen:

In [8]:
import sys
print(sys.getrecursionlimit())
sys.setrecursionlimit(10**6)

3000


Meist etwas schneller ist die nicht-rekursive Variante, die **Tabulation**. Die Ergebnisse, die die Rekursion von oben
nach unten (top-down) liefert, werden in einer Tabelle von unten nach oben (bottom-up) aufgebaut. Die Tabelle wird
genauso indiziert wie die Funktion `dp`. Wichtig ist die Reihenfolge: Jedes Teilproblem muss berechnet sein, bevor ein
größeres es braucht. Bei Suffixen geht die Schleife deshalb rückwärts.

Frog 1 als Tabelle: Zuerst tragen wir die Werte der Rekursionsbremse ein, dann den Rest.

In [9]:
a = [30, 10, 60, 10, 60, 50]
n = len(a)
dp = [0] * n
dp[n - 1] = 0
dp[n - 2] = abs(a[n - 1] - a[n - 2])
for i in range(n - 3, -1, -1):
    dp[i] = min(abs(a[i] - a[i + 1]) + dp[i + 1], abs(a[i] - a[i + 2]) + dp[i + 2])
print(dp)
print(dp[0])

[40, 40, 10, 40, 10, 0]
40


Cooldown als Tabelle mit zwei Indizes:

In [10]:
a = [1, 2, 3, 0, 2]
n = len(a)
dp = [[0] * 3 for i in range(n + 1)]       # dp[n][k] = 0 ist die Rekursionsbremse
for i in range(n - 1, -1, -1):
    for k in range(3):
        if k == 0:
            dp[i][k] = max(dp[i + 1][0], dp[i + 1][2] - a[i])
        elif k == 1:
            dp[i][k] = dp[i + 1][0]
        else:
            dp[i][k] = max(dp[i + 1][2], dp[i + 1][1] + a[i])
print(dp[0][0])

3


### Parent Pointer

Parent Pointer nutzen wir, wenn uns außer dem Ergebnis auch der Weg zum Ergebnis interessiert. Für jedes Teilproblem
merken wir uns in `prev`, welches kleinere Teilproblem das beste Ergebnis geliefert hat.

<img src='dp4.png' width='500'>

Beim Frosch zeigt `prev[i]` auf die Zahl, zu der der erste Sprung von i aus geht:

In [11]:
def dp(i):
    if i in memo:
        return memo[i]
    if i == n - 1:
        prev[i] = None
        return 0
    res = abs(a[i] - a[i + 1]) + dp(i + 1)
    prev[i] = i + 1
    if i + 2 < n and abs(a[i] - a[i + 2]) + dp(i + 2) < res:
        res = abs(a[i] - a[i + 2]) + dp(i + 2)
        prev[i] = i + 2
    memo[i] = res
    return res

a = [30, 10, 60, 10, 60, 50]
n = len(a)
memo = {}
prev = {}
print(dp(0))
weg = []
i = 0
while i is not None:
    weg.append(i)
    i = prev[i]
print('Weg über die Indizes', weg)

40
Weg über die Indizes [0, 2, 4, 5]


Hier steht die Prüfung `i + 2 < n` in der Rekursion, deshalb reicht die Bremse `i == n - 1`.

Bei Cooldown besteht ein Teilproblem aus dem Paar `(i, k)`. Der Parent Pointer zeigt auf das Paar, mit dem es weitergeht.
Aus der Folge der Zustände lesen wir ab, was an jedem Tag getan wird.

In [12]:
def dp(i, k):
    if (i, k) in memo:
        return memo[(i, k)]
    if i == n:
        prev[(i, k)] = None
        return 0
    if k == 0:
        res = dp(i + 1, 0)
        prev[(i, k)] = (i + 1, 0)
        if dp(i + 1, 2) - a[i] > res:
            res = dp(i + 1, 2) - a[i]
            prev[(i, k)] = (i + 1, 2)
    elif k == 1:
        res = dp(i + 1, 0)
        prev[(i, k)] = (i + 1, 0)
    else:
        res = dp(i + 1, 2)
        prev[(i, k)] = (i + 1, 2)
        if dp(i + 1, 1) + a[i] > res:
            res = dp(i + 1, 1) + a[i]
            prev[(i, k)] = (i + 1, 1)
    memo[(i, k)] = res
    return res

a = [48, 12, 60, 93, 97, 42, 25, 64, 17, 56, 85, 93, 9, 48, 52, 42, 58, 85, 81, 84, 69, 36, 1, 54, 23, 15, 72, 15, 11, 94]
n = len(a)
memo = {}
prev = {}
print('maximaler Gewinn:', dp(0, 0))

path = []
state = (0, 0)
while state is not None:
    path.append(state)
    state = prev[state]
print(path)

maximaler Gewinn: 428
[(0, 0), (1, 0), (2, 2), (3, 2), (4, 2), (5, 1), (6, 0), (7, 0), (8, 0), (9, 2), (10, 2), (11, 1), (12, 0), (13, 2), (14, 1), (15, 0), (16, 2), (17, 2), (18, 1), (19, 0), (20, 0), (21, 0), (22, 0), (23, 2), (24, 1), (25, 0), (26, 2), (27, 1), (28, 0), (29, 2), (30, 1)]


Wir übersetzen die Abfolge der Zustände in Aktionen. Ein Wechsel von Zustand 0 nach 2 heißt kaufen (K), von 2 nach 1
verkaufen (V), von 1 nach 0 ist ein Cooldown-Tag (C), sonst passiert nichts (x).

In [13]:
aktionen = []
for i in range(len(path) - 1):
    k1 = path[i][1]
    k2 = path[i + 1][1]
    if (k1, k2) == (0, 2):
        aktionen.append('K')
    elif (k1, k2) == (2, 1):
        aktionen.append('V')
    elif (k1, k2) == (1, 0):
        aktionen.append('C')
    else:
        aktionen.append('x')

gewinn = 0
for x, y in zip(a, aktionen):
    if y == 'K':
        gewinn -= x
    if y == 'V':
        gewinn += x
    print(x, y)
print('Gewinn aus den Aktionen:', gewinn)

48 x
12 K
60 x
93 x
97 V
42 C
25 x
64 x
17 K
56 x
85 V
93 C
9 K
48 V
52 C
42 K
58 x
85 V
81 C
84 x
69 x
36 x
1 K
54 V
23 C
15 K
72 V
15 C
11 K
94 V
Gewinn aus den Aktionen: 428


Zum Schluss rechnen wir den Gewinn aus den Aktionen nach, er stimmt mit `dp(0, 0)` überein.

#### Zum Weiterüben

- [atcoder Educational DP Contest](https://atcoder.jp/contests/dp/tasks): Frog 1, Frog 2 und Vacation sind die ersten
  Aufgaben dort.

Codingame:

- [Ways to make change](https://www.codingame.com/training/medium/ways-to-make-change)
- [Chained Matrix Products](https://www.codingame.com/training/medium/chained-matrix-products)
- [Number of Paths](https://www.codingame.com/training/medium/number-of-paths-between-2-points)
- [The Grand Festival I](https://www.codingame.com/training/medium/the-grand-festival---i)
- [The Grand Festival II](https://www.codingame.com/training/medium/the-grand-festival---ii)
- [Goro want Chocolate](https://www.codingame.com/training/medium/goro-want-chocolate)
- [Longest Increasing Subsequence](https://www.codingame.com/training/medium/longest-increasing-subsequence)
- [Rod cutting problem](https://www.codingame.com/training/medium/rod-cutting-problem)

Weitere Beispiele stehen im Notebook `dp_beispiele.ipynb`.